# 02 — Cleaning & Feature Engineering

Dari data mentah → frame siap-latih. Urutan: **gap flag → drop outlier → fitur
leakage-free → split loop-aware → encoder fit-di-train**. Menghasilkan
`outputs/training_ready.parquet` dengan kolom wajib `is_gap_suspected` + `deviation_ratio`.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
from src.data import load_raw, add_gap_flag, clean_outliers, time_split, sequence_ready
from src import features as F, models as Mdl

raw = load_raw()
flagged = add_gap_flag(raw)                       # is_gap_suspected + loop_n_segments (RAW order)
clean, report = clean_outliers(flagged)          # drop > 3600s (D3)
report

[clean_outliers] dropped 2994 rows (0.853%) with traveling_time_sec > 3600s; 348109 rows remain.


{'rows_before': 351103,
 'rows_after': 348109,
 'rows_dropped': 2994,
 'pct_dropped': 0.853,
 'max_sec_threshold': 3600}

## Fitur leakage-free (dihitung di frame penuh, time-ordered, di-shift)
`baseline_segment_hour` = expanding mean per (segment,hour) **shifted** = pengganti bersih
`average_time_sec`. `rolling_mean_segment_5`, `trip_progress`, cyclic time, dll.

In [2]:
feat = F.build_base_features(clean)
new_cols = ["hour", "is_weekend", "is_rush_hour", "hour_sin", "trip_progress",
            "baseline_segment_hour", "rolling_mean_segment_5", "deviation_ratio"]
feat[new_cols].head()

,hour,is_weekend,is_rush_hour,hour_sin,trip_progress,baseline_segment_hour,rolling_mean_segment_5,deviation_ratio
300556,0,1,0,0.0,0.952381,88.41900,88.41900,0.482638
300557,0,1,0,0.0,0.882353,86.90500,86.90500,0.811967
300555,0,1,0,0.0,0.894737,76.88150,76.88150,0.985984
300560,0,1,0,0.0,0.947368,67.87500,67.87500,1.029714
300559,0,1,0,0.0,0.941176,66.86875,66.86875,0.909438


## Split loop-aware time-based (D6) + encoder fit-di-train (anti-leakage)
Tiap `no_do` utuh di satu sisi (tak straddle) → Loop MAE tak rusak, tak bocor antar-split.

In [3]:
tr, te = time_split(feat)
art = F.fit_feature_artifacts(tr)                 # volatility, kategori, baseline-model: TRAIN saja
tr = F.transform_with_artifacts(tr, art)
te = F.transform_with_artifacts(te, art)
print(f"train {len(tr)} ({tr['departure_time'].min().date()}..{tr['departure_time'].max().date()}) | "
      f"test {len(te)} ({te['departure_time'].min().date()}..{te['departure_time'].max().date()})")
print("overlap loop train∩test (harus 0):", len(set(tr['no_do']) & set(te['no_do'])))

train 258590 (2026-02-01..2026-02-22) | test 89519 (2026-02-22..2026-02-28)
overlap loop train∩test (harus 0): 0


## Bukti TIDAK ada leakage
XGBoost jujur (15 fitur, tanpa `average_time_sec`): **median AE ~14.5s** — kalau ada fitur
yang membocorkan target, median AE jatuh ke ~0. `baseline_segment_hour` korelasi 0.69 (wajar).

In [4]:
Xtr, ytr_log, _ = Mdl.make_xy(tr); Xte, _, yte = Mdl.make_xy(te)
m = Mdl.train_xgb(Xtr, ytr_log)
pred = Mdl.predict_sec(m, Xte)
print(f"median AE = {np.median(np.abs(yte - pred)):.2f}s  (jujur, bukan ~0)")
print(f"corr(baseline_segment_hour, target) = {tr[['baseline_segment_hour','traveling_time_sec']].corr().iloc[0,1]:.3f}")
print("FEATURE_COLS:", F.FEATURE_COLS)
print("output-only (mengandung target, BUKAN fitur):", F.OUTPUT_ONLY_COLS)

median AE = 14.51s  (jujur, bukan ~0)
corr(baseline_segment_hour, target) = 0.693
FEATURE_COLS: ['hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'hour_sin', 'hour_cos', 'stop_sequence', 'trip_progress', 'loop_n_segments', 'is_gap_suspected', 'baseline_segment_hour', 'rolling_mean_segment_5', 'segment_volatility', 'segment_id_code', 'trip_id_code']
output-only (mengandung target, BUKAN fitur): ['deviation_ratio', 'deviation_ratio_clean']


## Kolom output wajib + simpan training-ready dataframe

In [5]:
tr["split"], te["split"] = "train", "test"
ready = pd.concat([tr, te]).sort_values(["no_do", "stop_sequence"]).reset_index(drop=True)
print("is_gap_suspected:", ready["is_gap_suspected"].dtype,
      "| deviation_ratio terisi:", f"{ready['deviation_ratio'].notna().mean()*100:.1f}%")
ready.to_parquet("../outputs/training_ready.parquet", index=False)
print("saved -> outputs/training_ready.parquet", ready.shape)
ready[["no_do", "stop_sequence", "segment_id", "traveling_time_sec",
       "is_gap_suspected", "deviation_ratio", "baseline_segment_hour"]].head()

is_gap_suspected: bool | deviation_ratio terisi: 100.0%


saved -> outputs/training_ready.parquet (348109, 29)


,no_do,stop_sequence,segment_id,traveling_time_sec,is_gap_suspected,deviation_ratio,baseline_segment_hour
0,DO-DUMMY-00003758,21,SEG-DUMMY-009393,295.884,False,1.461620,505.076333
1,DO-DUMMY-00003785,17,SEG-DUMMY-009422,49.862,False,0.985984,76.881500
2,DO-DUMMY-00003785,18,SEG-DUMMY-009049,63.850,False,1.029714,67.875000
3,DO-DUMMY-00003785,19,SEG-DUMMY-008872,359.084,False,0.918670,68.569167
4,DO-DUMMY-00003786,13,SEG-DUMMY-009353,71.545,True,0.888217,95.273722
